# Zero-Knowledge Proofs

TEL252 · Criptografía y Seguridad en la Información

Durante la incorporación de un dispositivo, el celular quiere comprobar que el dispositivo conoce el secreto asociado a un valor público. El dispositivo puede hacer la misma comprobación sobre el celular cuando se invierten los roles.

## 1. ¿Qué es Zero-Knowledge Proofs?

Es un protocolo que permite una afirmación sin revelar información adicional sobre el secreto. En Schnorr, la afiración vendria a ser "conozco un $x$ tal que $Y=g^x$"

- **Publico.** Ambos conocen el grupo cíclico de orden primo $q$, su generador $g$ y el valor $Y$.
- **Privado.** El probador conoce $x$. Recuperar $x$ a partir de $g$ e $Y$ es un problema de logaritmo discreto.

Para entender qué se espera de la prueba, conviene distinguir tres propiedades.

| Propiedad | Qué significa en este ejemplo |
| --- | --- |
| Completitud | Si el dispositivo conoce $x$ y ambos siguen el protocolo, el celular acepta la prueba. |
| Solidez de conocimiento | Un dispositivo que logra responder correctamente a desafíos impredecibles con suficiente probabilidad demuestra conocimiento de $x$. |
| Conocimiento cero | Un verificador honesto puede simular una conversación con la misma distribución sin conocer $x$. Por eso, ver una conversación válida no le aporta información adicional sobre $x$. |

La simulación puede elegir el desafío y la respuesta primero y calcular el compromiso después. En el intercambio real el orden es al revés: el dispositivo envía el compromiso antes de que el celular elija el desafío. Esa diferencia permite que una conversación simulada tenga la forma correcta sin que el simulador pueda responder a un desafío que recibe después de comprometerse. [ZKDocs, protocolo de identificación de Schnorr](https://www.zkdocs.com/docs/zkdocs/zero-knowledge-protocols/schnorr/) · [RFC 8235, sección 2](https://www.rfc-editor.org/rfc/rfc8235.html#section-2)

#### Identificación de Schnorr

Para llevar la misma idea a curvas elípticas cambiamos de notación multiplicativa a notación aditiva. El generador $g$ pasa a escribirse como un punto base $G$ de orden $q$, y la operación $g^x$ corresponde a la multiplicación escalar $xG$.

En lo que sigue, el dispositivo conserva $x$ en privado y publica el punto $Y=xG$. Para demostrar que conoce $x$, se intercambian tres mensajes.

| Paso | Quién envía | Qué envía |
| --- | --- | --- |
| Compromiso | Dispositivo | Elige un numero random $v$ y envía $V=vG$. |
| Desafío | Celular | Después de recibir $V$, elige un desafío aleatorio $c$ y lo envía. |
| Respuesta | Dispositivo | Envía el escalar $r=(v-cx)\bmod q$. |

El celular verifica que $V=rG+cY$. Si el dispositivo siguió el protocolo, la ecuación se cumple porque

$$
rG+cY=(v-cx)G+c(xG)=vG=V.
$$

El orden importa. El celular elige $c$ después de recibir $V$, de modo que el dispositivo debe comprometerse antes de conocer el desafío. El secreto $x$ se usa para calcular la respuesta y queda en el dispositivo.

Con números pequeños para seguir la cuenta, si $q=11$, $x=3$, $v=7$ y $c=2$, entonces $r=1$. El celular verifica que $1G+2(3G)=7G=V$. Estos números son solo ilustrativos. [RFC 8235, sección 2](https://www.rfc-editor.org/rfc/rfc8235.html#section-2)

In [ ]:
# Versión básica · Identificación de Schnorr
# Grupo didáctico Z/11Z. Un entero k representa el punto kG, con G = 1.
# Los valores fijos reproducen el ejemplo de arriba; son públicos en esta demo.
q = 11
G = 1
x = 3
Y = (x * G) % q

# 1. Dispositivo → celular: compromiso
v = 7
V = (v * G) % q
print(f"1. Compromiso V = {V}G")

# 2. Celular → dispositivo: desafío
c = 2
print(f"2. Desafío c = {c}")

# 3. Dispositivo → celular: respuesta
r = (v - c * x) % q
print(f"3. Respuesta r = {r}")

# El celular verifica usando únicamente datos públicos.
punto_verificado = (r * G + c * Y) % q
print(f"Verificación: {V}G = {punto_verificado}G")
assert V == punto_verificado


#### non-interactive ZKP

En el protocolo anterior el celular tiene que contestar con un desafío antes de que el dispositivo pueda terminar la prueba. Una **prueba no interactiva** permite que el dispositivo envíe una prueba completa en un solo mensaje y que el celular la verifique localmente. Si ambos actores demuestran conocimiento, cada uno envía su propia prueba completa.

La transformación de **Fiat–Shamir** sustituye el desafío que enviaba el celular por uno derivado de un hash criptográfico. El dispositivo genera $V=vG$, calcula $c$ con el hash y luego calcula la respuesta $r=(v-cx)\bmod q$. Envía $Y$ junto con la prueba $(V,r)$. El celular vuelve a calcular $c$ a partir de los mismos datos públicos y comprueba $V=rG+cY$.

El hash cumple dos funciones. Primero, permite que ambas partes obtengan el mismo desafío de manera independiente. Segundo, vincula el desafío a los datos incluidos en su entrada. Por ejemplo, incluir $V$ hace que cambiar el compromiso cambie normalmente el desafío. La misma idea se aplica a los datos de contexto, como el identificador de quien presenta la prueba. [RFC 8235, secciones 1 y 3.3](https://www.rfc-editor.org/rfc/rfc8235.html#section-3.3)

#### Vulnerabilidad

Consideremos una variante que usa $c=H(G,V,Y)$. El dispositivo envía su identificador junto con $Y$ y $(V,r)$, pero el identificador queda fuera del cálculo. Si el celular usa ese desafío y solo comprueba la ecuación de Schnorr, ocurre lo siguiente.

| Mensaje recibido | Desafío que calcula el celular | Resultado de la ecuación |
| --- | --- | --- |
| Identificador `sensor-A`, punto $Y$, prueba $(V,r)$ | $H(G,V,Y)$ | Se cumple. |
| Identificador `sensor-B`, mismo punto $Y$, misma prueba $(V,r)$ | $H(G,V,Y)$ | Se sigue cumpliendo. |

Al cambiar solo el identificador, los valores de la ecuación $V=rG+cY$ quedan iguales. Por eso, la prueba puede presentarse con otro `UserID` aunque el emisor original solo la haya enviado como `sensor-A`. Esta falla corresponde a la variante no interactiva descrita aquí.

La RFC 8235 incluye el `UserID` en el hash del desafío. El celular verifica la prueba usando el identificador que corresponde al intercambio. Si se presenta la misma prueba con otro identificador, el desafío calculado normalmente cambia y la ecuación deja de cumplirse. Vincular la prueba a un identificador todavía deja abierta la pregunta de cómo sabe el celular que ese identificador corresponde al sensor físico que el usuario quiere incorporar. [RFC 8235, sección 3.3](https://www.rfc-editor.org/rfc/rfc8235.html#section-3.3)

In [ ]:
# Vulnerabilidad · Ejecutar después de la celda de Schnorr interactivo
from hashlib import sha256


def desafio_sin_user_id(generador, compromiso, valor_publico):
    # Un byte por elemento es suficiente para el grupo de juguete.
    datos = bytes([generador, compromiso, valor_publico])
    return int.from_bytes(sha256(datos).digest(), "big")


def verificar_mensaje(mensaje):
    valor_publico = mensaje["Y"]
    compromiso, respuesta = mensaje["prueba"]
    desafio = desafio_sin_user_id(G, compromiso, valor_publico)
    valido = compromiso == (respuesta * G + desafio * valor_publico) % q
    return desafio, valido


# El dispositivo crea una prueba no interactiva con otro compromiso.
v_prueba = 8
V_prueba = (v_prueba * G) % q
c_prueba = desafio_sin_user_id(G, V_prueba, Y)
r_prueba = (v_prueba - c_prueba * x) % q
original = {"UserID": "sensor-A", "Y": Y, "prueba": (V_prueba, r_prueba)}

# Se modifica únicamente la etiqueta del mensaje.
reetiquetado = {**original, "UserID": "sensor-B"}

c_original, acepta_original = verificar_mensaje(original)
c_reetiquetado, acepta_reetiquetado = verificar_mensaje(reetiquetado)

for mensaje, desafio, acepta in (
    (original, c_original, acepta_original),
    (reetiquetado, c_reetiquetado, acepta_reetiquetado),
):
    resultado = "aceptada" if acepta else "rechazada"
    print(f"{mensaje['UserID']}: desafío = {desafio:#066x}, prueba {resultado}")

assert c_original == c_reetiquetado
assert acepta_original and acepta_reetiquetado
